# 7.3.1 — Supervised Learning: More Models

**Learning goals**
- Train and evaluate a linear regression model.
- Explain why logistic regression is a *classification* algorithm.
- Train, visualize, and evaluate a decision tree.
- Identify signs of underfitting and overfitting.

**Previously:** KNN, train/test split, regression and classification metrics, scaling, and `Pipeline`.  
**Next:** ensemble methods, starting from decision trees.

We will use **California Housing** for regression and **Iris** for classification.

## 1. Regression: California Housing

We will reuse a familiar dataset and focus on the *models*, rather than repeating all feature-engineering steps. As in the original lesson, we keep five features (`MedInc`, `HouseAge`, `AveRooms`, `Latitude`, `Longitude`) so that the comparison remains manageable.

**Reminder:** split the data before fitting any preprocessing operation.

**Feature selection reminder:** Last session we used correlation analysis to explore relationships with the target. For a linear model, correlations are a useful starting point, but they do not capture the joint contribution of several predictors. Correlated predictors can also introduce **multicollinearity**, making individual coefficients harder to interpret. The five features below are chosen **for illustration**, not because they form an optimal subset. We will not repeat the correlation matrix here.
**Geographical features:** `Latitude` and `Longitude` can help predict housing values because location matters. However, geographical effects are often nonlinear, and coordinates alone cannot explain all price differences. Two homes at nearly the same location may have different prices because of features we have not measured. **Dataset detail:** California Housing records describe census block groups (including median house value), not individual homes. We include coordinates as useful location proxies, not as unique identifiers or causal explanations.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
)

RANDOM_STATE = 42

In [ ]:
housing = fetch_california_housing(as_frame=True)
X_housing = housing.data[["MedInc", "HouseAge", "AveRooms", "Latitude", "Longitude"]]
y_housing = housing.target

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_housing, y_housing, test_size=0.20, random_state=RANDOM_STATE
)

print("Train:", X_train_reg.shape, "Test:", X_test_reg.shape)
X_train_reg.head()

## 2. Linear Regression

A linear regression model predicts a **continuous** target using an additive combination of the input features:

$$\hat{y}=\beta_0+\beta_1x_1+\cdots+\beta_px_p$$

Ordinary least squares estimates the coefficients by minimizing the **sum of squared residuals** on the training data. This is not the same as minimizing absolute distances.

We place scaling inside a `Pipeline` to keep preprocessing and prediction together. Linear regression does not inherently require scaling for its predictions, but scaling makes the coefficient magnitudes easier to compare (though not a universal measure of importance).

In [ ]:
linear_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

linear_pipeline.fit(X_train_reg, y_train_reg)
y_pred_test_linear = linear_pipeline.predict(X_test_reg)

linear_metrics = {
    "MAE": mean_absolute_error(y_test_reg, y_pred_test_linear),
    "RMSE": np.sqrt(mean_squared_error(y_test_reg, y_pred_test_linear)),
    "R2": r2_score(y_test_reg, y_pred_test_linear),
}
pd.Series(linear_metrics, name="Linear Regression")

The target is measured in units of **$100,000**. For example, an MAE of `0.5` corresponds to about $50,000.

Let's compare predictions with the observed values. Perfect predictions would lie on the diagonal.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test_reg, y_pred_test_linear, s=9, alpha=0.25)
limits = [min(y_test_reg.min(), y_pred_test_linear.min()),
          max(y_test_reg.max(), y_pred_test_linear.max())]
ax.plot(limits, limits, "k--", label="Perfect prediction")
ax.set(xlabel="Actual house value ($100,000)",
       ylabel="Predicted house value ($100,000)",
       title="Linear Regression: actual vs predicted")
ax.legend()
plt.show()

### Interpreting coefficients

Because we standardized the features, a coefficient describes the change in the prediction associated with a **one-standard-deviation increase** in that feature, holding the others constant.

Be careful: coefficients can be unstable when features are correlated; they do **not** automatically establish causation.

To rank coefficients by magnitude while **preserving their signs**, we will sort by absolute value and draw a horizontal bar chart. This is a comparison of *standardized coefficients*, not a definitive ranking of predictive importance.
In the chart below, **blue bars indicate positive coefficients** and **red bars indicate negative coefficients**. Bar length indicates absolute magnitude; direction and color preserve the sign.


In [ ]:
linear_model = linear_pipeline.named_steps["model"]
coefficients = pd.Series(
    linear_model.coef_, index=X_train_reg.columns, name="Standardized coefficient"
)

# Sort by absolute magnitude, but retain the original signed coefficients.
coefficients_sorted = coefficients.reindex(coefficients.abs().sort_values().index)

fig, ax = plt.subplots(figsize=(7, 4))
# Keep the coefficient sign while using absolute magnitude for ordering.
bar_colors = ["red" if coefficient < 0 else "blue" for coefficient in coefficients_sorted]
coefficients_sorted.plot.barh(ax=ax, color=bar_colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(xlabel="Coefficient (target units per 1 SD of feature)",
       title="Linear Regression: standardized coefficients")
plt.tight_layout()
plt.show()

print("Intercept:", round(linear_model.intercept_, 3))
coefficients.reindex(coefficients.abs().sort_values(ascending=False).index).to_frame()

**Check for understanding (1 minute)**

1. Why do we use `fit()` only on training data?
2. If a coefficient is negative, does that prove that the feature *causes* house prices to fall?

## 3. Logistic Regression: Classification with Iris

Despite its name, **Logistic Regression is used for classification**. Like linear regression, it begins with a linear combination of features. In binary classification, a **sigmoid** transforms this score into a probability between 0 and 1:

$$p(y=1\mid x)=\frac{1}{1+e^{-z}},\qquad z=\beta_0+\beta_1x_1+\cdots+\beta_px_p$$

With the usual threshold of `0.5`, probabilities at or above the threshold are assigned to class 1. Unlike linear regression, logistic regression learns its parameters using a classification loss (log loss).

We'll first solve an easy **binary** task: *setosa* versus *not setosa*.

**Why `max_iter=1000`?** Logistic regression estimates its coefficients through an iterative optimization procedure. `max_iter` is the **maximum number of iterations allowed**, not the number it must perform. If the optimizer has not converged by the default limit, scikit-learn may raise a `ConvergenceWarning`. Increasing the limit can help, but we should also check scaling and other convergence issues.

In [ ]:
iris = load_iris(as_frame=True)
X_iris = iris.data
y_iris_binary = (iris.target == 0).astype(int)  # 1 = setosa; 0 = other species

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_iris, y_iris_binary, test_size=0.25,
    stratify=y_iris_binary, random_state=RANDOM_STATE
)

logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
logistic_pipeline.fit(X_train_cls, y_train_cls)

y_pred_test_logistic = logistic_pipeline.predict(X_test_cls)
y_proba_test_logistic = logistic_pipeline.predict_proba(X_test_cls)[:, 1]

print("Accuracy:", round(accuracy_score(y_test_cls, y_pred_test_logistic), 3))
print(classification_report(y_test_cls, y_pred_test_logistic,
                            target_names=["not setosa", "setosa"]))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test_cls, y_pred_test_logistic,
    display_labels=["not setosa", "setosa"], cmap="Blues"
)
plt.title("Logistic Regression: confusion matrix")
plt.show()

pd.DataFrame({
    "Actual class": y_test_cls.to_numpy()[:8],
    "P(setosa)": y_proba_test_logistic[:8].round(3),
    "Predicted class": y_pred_test_logistic[:8],
})

### Beyond binary classification

`LogisticRegression` also supports **multiclass classification**. We can train it on all three Iris species with almost the same code.

In [ ]:
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    iris.data, iris.target, test_size=0.25,
    stratify=iris.target, random_state=RANDOM_STATE
)

multiclass_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
multiclass_pipeline.fit(X_train_multi, y_train_multi)
y_pred_test_multi = multiclass_pipeline.predict(X_test_multi)
print("Multiclass accuracy:", round(accuracy_score(y_test_multi, y_pred_test_multi), 3))

**Check for understanding (1 minute)**

1. Why is Logistic Regression a classifier even though its name contains “regression”?
2. What information do we lose when we use `predict()` instead of `predict_proba()`?

## 4. Decision Trees: Learning a Sequence of Rules

A decision tree repeatedly splits the data using questions such as **`MedInc <= 5.0`**. Each terminal node (leaf) produces a prediction. For regression, a leaf typically predicts the mean target value of the training examples in that leaf.

Trees can model nonlinear relationships and **do not require feature scaling**: their splits depend on feature thresholds, not on distances between samples.

**Each model can have its own pipeline.** A `Pipeline` can even contain **only the final estimator**, without any transformer. We will explicitly construct one for the tree so that all models use the same `fit()` / `predict()` interface.

Let's begin with a shallow tree that we can visualize.

In [ ]:
tree_pipeline = Pipeline([
    ("model", DecisionTreeRegressor(max_depth=3, random_state=RANDOM_STATE)),
])
tree_pipeline.fit(X_train_reg, y_train_reg)

# Retrieve the fitted estimator from the pipeline for visualization.
tree_small = tree_pipeline.named_steps["model"]
fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(
    tree_small, feature_names=X_train_reg.columns,
    filled=True, rounded=True, fontsize=8, ax=ax
)
ax.set_title("Decision Tree Regressor (max_depth=3)")
plt.show()

### Underfitting and overfitting

`max_depth` limits how many levels the tree can grow. A shallow tree may be too simple (**underfitting**). A very deep tree may learn peculiarities of the training set (**overfitting**).

There is **no universal rule** requiring tree depth to be smaller than the number of features: a tree can split on the same feature at different levels.

We will use a few fixed depths to **observe** the effect, not to select the best hyperparameter. Systematic tuning with cross-validation comes tomorrow.

In [ ]:
tree_results = []
for depth in [2, 3, 6, 12, None]:
    # This pipeline has no preprocessing transformer: the tree needs none.
    depth_pipeline = Pipeline([
        ("model", DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE)),
    ])
    depth_pipeline.fit(X_train_reg, y_train_reg)
    y_pred_train_tree = depth_pipeline.predict(X_train_reg)
    y_pred_test_tree = depth_pipeline.predict(X_test_reg)
    tree_results.append({
        "max_depth": str(depth),
        "train MAE": mean_absolute_error(y_train_reg, y_pred_train_tree),
        "test MAE": mean_absolute_error(y_test_reg, y_pred_test_tree),
    })

tree_results_df = pd.DataFrame(tree_results)
tree_results_df.round(3)

In [ ]:
ax = tree_results_df.plot(
    x="max_depth", y=["train MAE", "test MAE"], marker="o", figsize=(7, 4)
)
ax.set(xlabel="Maximum tree depth", ylabel="MAE (lower is better)",
       title="Model complexity: training vs test error")
ax.grid(alpha=0.2)
plt.show()

**Important:** We inspected the test results here to *illustrate* overfitting, not to choose a final depth. In a real project, choose hyperparameters using cross-validation on the training set and reserve the test set for the final evaluation.

### Which features does the tree use?

Tree-based `feature_importances_` summarizes the reduction in the splitting criterion attributed to each feature. It is useful for exploration, but it is not causal importance and can be biased.

### Feature importance: linear model versus tree

The two bar charts answer **different questions**:

- **Linear Regression:** signed coefficients show the direction and magnitude of an association *after scaling*, holding other predictors constant. We sort by absolute value but keep the sign.
- **Decision Tree:** `feature_importances_` measures the relative reduction in the tree's splitting criterion attributed to each feature. Values are nonnegative and sum to 1 (when splits occur); they do **not** show direction. These impurity-based importances can be misleading, especially for features with many possible split points.

Neither chart proves causation or provides a universally reliable ranking of predictive usefulness.

In [ ]:
# Unlike linear coefficients, tree importances are nonnegative.
# Sort from least to most important so the largest bar appears at the top.
tree_importances = pd.Series(
    tree_small.feature_importances_, index=X_train_reg.columns,
    name="Tree feature importance"
)
tree_importances_sorted = tree_importances.sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
tree_importances_sorted.plot.barh(ax=ax)
ax.set(xlabel="Impurity-based feature importance",
       title="Decision Tree: feature importances (max_depth=3)")
plt.tight_layout()
plt.show()

tree_importances.sort_values(ascending=False).to_frame()

**Check for understanding (2 minutes)**

1. Why does a decision tree not need `StandardScaler`?
2. If training error decreases while test error increases, what might be happening?
3. Can the same feature appear at several levels of one tree?

## 5. Wrap-up: Three Different Models

| Model | Task | Core idea | Main consideration |
|---|---|---|---|
| Linear Regression | Regression | Additive linear relationship | May underfit nonlinear patterns |
| Logistic Regression | Classification | Linear score transformed into class probabilities | Linear decision boundaries in the original feature space |
| Decision Tree | Regression or classification | Sequence of feature-based splits | Can overfit if left unrestricted |

**Key takeaway:** A fair modeling comparison considers the **selected features + preprocessing pipeline + estimator (and its hyperparameters)**. Different algorithms may require different preprocessing; compare complete approaches on consistent data splits and appropriate metrics. We will revisit this idea when tuning models with cross-validation on Thursday.

**Next notebook — 7.3.2 Ensemble Methods:** Instead of relying on one decision tree, what happens when we combine many models? We'll explore Random Forest, Bagging, AdaBoost, and Gradient Boosting.

**Optional exploration:** [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) · [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) · [DecisionTreeRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)